In [1]:
import numpy as np
import pandas as pd
import xarray as xr

DATA = "../DATA"
STNS = np.loadtxt(f"{DATA}/Stations.txt", dtype="U2")
VNS = np.loadtxt(f"{DATA}/Products.txt", dtype="<U10")
VNS = VNS[VNS != "TH"]            
PRODUCTS = [str(v) for v in VNS if v != "PM"]

In [2]:
def load(stn, vn):
    da = xr.open_dataarray(f"{DATA}/{stn}/{vn}_{stn}.nc")
    t = pd.to_datetime(np.ravel(da["Year"] * 1000 + da["DOY"]), format="%Y%j")
    return xr.DataArray(np.ravel(da, order="F"), coords={"datetime": t}, dims="datetime")


def taylor_skill(ref, x, r_max=1.0):
    """Eq. (10): S = 4(1+R)^4 / [(d + 1/d)^2 (1+R_max)^4], d = sigma_x / sigma_ref."""
    r = xr.corr(ref, x, dim="datetime")
    d = x.std(dim="datetime") / ref.std(dim="datetime")
    return 4 * (1 + r) ** 4 / ((d + 1 / d) ** 2 * (1 + r_max) ** 4)


def stats(ref, x):
    err = x - ref
    return pd.DataFrame({
        "RMSE": np.sqrt((err ** 2).mean(dim="datetime")).values,
        "CC": xr.corr(ref, x, dim="datetime").values,
        "Skill": taylor_skill(ref, x).values,
    }, index=ref["Station"].values)

In [3]:
ET = xr.concat(
    [xr.concat([load(stn, vn) for vn in VNS], dim="Product") for stn in STNS], dim="Station"
).assign_coords(Product=VNS, Station=STNS)
ET.attrs["units"] = "mm/8days"
PM = ET.sel(Product="PM")

In [4]:
S = taylor_skill(PM, ET.sel(Product=PRODUCTS))
W = S / S.sum(dim="Product")
STS = (W * ET.sel(Product=PRODUCTS)).sum(dim="Product")
STS.attrs = {"units": "mm/8days", "long_name": "STS merged ET"}
STS.to_netcdf(f"{DATA}/STN_STS.nc")

In [5]:
COM = xr.open_dataarray(f"{DATA}/STN_COM.nc")

sts, com = stats(PM, STS), stats(PM, COM)
summary = pd.DataFrame({"STS": sts.mean(), "COM": com.mean()})
print(summary.to_string(formatters={"STS": "{:.3f}".format, "COM": "{:.3f}".format}))
print()
print(f"COM lower RMSE:    {(com.RMSE < sts.RMSE).sum()} of {len(STNS)} stations")
print(f"COM higher skill:  {(com.Skill > sts.Skill).sum()} of {len(STNS)} stations")

        STS   COM
RMSE  2.326 1.456
CC    0.978 0.982
Skill 0.936 0.965

COM lower RMSE:    20 of 21 stations
COM higher skill:  20 of 21 stations
